# Qwen 3.5 9B — Qwen-only Kaggle benchmark

Enable **GPU and internet** in Kaggle. Attach `colab_benchmark_inputs.zip` as a private Kaggle Dataset through **Add Input**. The source snapshot includes our current local changes, without credentials, indexes, or model weights.

This runs a new **full-dataset Qwen-only benchmark**. Retrieval uses the original question, 25 dense + 25 sparse candidates, BGE, and **top-5 only**. There are no refinement calls. Qwen receives top-5 evidence and a **2,048-token output cap**. Results include nDCG, token usage, finish reasons, latency, and retries. README results and the previous experiment stay unchanged; Qwen is now the application default. Set --limit to 50 in the run cell for a smaller sample, or keep 0 for all questions.

Expect model setup/download time before inference. Qwen weights are approximately 6.55 GB. On two T4 GPUs, two independent workers process separate questions concurrently. Each GPU hosts Qwen, embeddings, and BGE. A single GPU must have enough memory for both.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, subprocess, sys, zipfile, shutil
REPO = Path('/kaggle/working/vietnamese-legal-chatbot')
REPO.mkdir(parents=True, exist_ok=True)
source = base64.b64decode('...')
assert hashlib.sha256(source).hexdigest() == 'c61d9e5c20130cafff6d0f9056adbb52bf5427ceffc5f9cb66416929e3cf5beb'
with zipfile.ZipFile(io.BytesIO(source)) as archive:
    archive.extractall(REPO)
os.chdir(REPO)
print('Current source snapshot restored.')

## Verify GPUs, install dependencies, and prepare the dataset

In [ ]:
gpu_probe = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, check=True)
GPUS = gpu_probe.stdout.strip().splitlines()
assert GPUS, 'Enable a Kaggle GPU accelerator.'
print(gpu_probe.stdout)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Notebook checks only; child workers pin their own GPUs.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'requirements-trial.txt'], check=True)
subprocess.run([sys.executable, 'download_dataset.py'], check=True)
import torch
assert torch.cuda.is_available(), 'PyTorch cannot see the GPU.'
print('PyTorch GPU:', torch.cuda.get_device_name(0))

## Restore the existing indexes from the attached input Dataset

In [ ]:
"""Accept Kaggle's ZIP or automatically unpacked Dataset layout."""
import hashlib
from pathlib import Path, PurePosixPath
import shutil
from zipfile import ZipFile, ZIP_STORED


def input_digest(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()


def restore_kaggle_inputs(input_root, repo, expected):
    input_root, repo = Path(input_root), Path(repo)
    bundle = next(input_root.rglob('colab_benchmark_inputs.zip'), None)
    destination = repo / 'colab_vector_index.zip'
    if bundle:
        if input_digest(bundle) != expected['bundle_sha256']:
            raise ValueError('Index bundle checksum differs.')
        with ZipFile(bundle) as archive:
            if set(archive.namelist()) != {'bm25_index.pkl', 'colab_vector_index.zip'}:
                raise ValueError('Unexpected bundle contents.')
            (repo / 'index').mkdir(parents=True, exist_ok=True)
            with archive.open('bm25_index.pkl') as source, (repo / 'index/bm25_index.pkl').open('wb') as target:
                shutil.copyfileobj(source, target)
            archive.extract('colab_vector_index.zip', repo)
        print('Verified original ZIP input.')
        return destination

    bm25 = next((p for p in input_root.rglob('bm25_index.pkl') if input_digest(p) == expected['bm25_sha256']), None)
    if bm25 is None:
        raise ValueError('Matching BM25 index not found in the attached Kaggle Dataset.')
    vector_files = expected['vector_files']
    for name in vector_files:
        path = PurePosixPath(name)
        if path.is_absolute() or '..' in path.parts or '\\' in name or ':' in name:
            raise ValueError('Unsafe expected vector path.')
    manifests = [p for p in input_root.rglob('build_manifest.json')
                 if input_digest(p) == vector_files['build_manifest.json']]
    vector_root = None
    for manifest in manifests:
        root = manifest.parent
        if all((root / name).is_file() and input_digest(root / name) == checksum
               for name, checksum in vector_files.items()):
            vector_root = root
            break
    if vector_root is None:
        raise ValueError('Complete matching vector files not found; the extracted index is incomplete or changed.')
    # Verify all input hashes before changing any destination files.
    (repo / 'index').mkdir(parents=True, exist_ok=True)
    shutil.copyfile(bm25, repo / 'index/bm25_index.pkl')
    with ZipFile(destination, 'w', compression=ZIP_STORED) as archive:
        for name in vector_files:
            archive.write(vector_root / name, name)
    print('Verified automatically extracted Kaggle input and rebuilt the portable archive.')
    return destination


import os, subprocess, sys
EXPECTED_INPUTS = {'bundle_sha256': '9509bbe6b93c0fca3e41458f87c7b7b422ac803b87ff2c9babb2a918caae5d0e', 'bm25_sha256': 'b6dfc625509c717e0a8d213da26e61854945688b2299a133bd4a677e3e50f523', 'vector_files': {'build_manifest.json': '97b1e971626e6919213401a1228c3ae081fce614839afa8367cca48ae061398f', 'qdrant/meta.json': '77feeada277abff14b1a52c634379e3958f8e71ebb7db01839160dc359049239', 'qdrant/collection/bkai_biencoder_vietnamese_legal_corpus/storage.sqlite': 'dd6f58a96046140edee0cf9359086508de25f4fb3c1c26b9ff67998739c59f01'}}
archive_path = restore_kaggle_inputs('/kaggle/input', REPO, EXPECTED_INPUTS)
subprocess.run([sys.executable, 'import_vector_index.py', str(archive_path)], cwd=REPO, check=True,
               env={**os.environ, 'QDRANT_PATH': 'index/qdrant', 'QDRANT_URL': '', 'QDRANT_API_KEY': ''})
print('Existing indexes restored; no re-embedding.')

## Start Qdrant 1.19.0 and load the same vectors

In [ ]:
import requests, tarfile, time, urllib.request
RUNTIME = REPO / 'qdrant-runtime'
RUNTIME.mkdir(exist_ok=True)
BINARY_DIR = Path('/tmp/legal-rag-qdrant-bin')
BINARY_DIR.mkdir(parents=True, exist_ok=True)
binary = BINARY_DIR / 'qdrant'
if not binary.exists():
    package = RUNTIME / 'qdrant.tar.gz'
    urllib.request.urlretrieve('https://github.com/qdrant/qdrant/releases/download/v1.19.0/qdrant-x86_64-unknown-linux-gnu.tar.gz', package)
    assert hashlib.sha256(package.read_bytes()).hexdigest() == 'e4405091f67d02f96fb941695ef8a6974e677632507ff7b04a3fcbb332ad9c19'
    with tarfile.open(package) as archive:
        member = next(m for m in archive.getmembers() if m.isfile() and Path(m.name).name == 'qdrant')
        with archive.extractfile(member) as source, binary.open('wb') as target:
            shutil.copyfileobj(source, target)
binary.chmod(0o755)
config = RUNTIME / 'config.yaml'
config.write_text('storage:\n  storage_path: ' + str(RUNTIME / 'storage') + '\nservice:\n  host: 127.0.0.1\n  http_port: 6333\n  grpc_port: 6334\n', encoding='utf-8')
try:
    running = requests.get('http://localhost:6333', timeout=2).json()
except requests.RequestException:
    running = None
if running is None:
    server_log = (RUNTIME / 'server.log').open('ab')
    server = subprocess.Popen([str(binary), '--config-path', str(config)], cwd=RUNTIME,
                              stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if server.poll() is not None:
            raise RuntimeError((RUNTIME / 'server.log').read_text()[-2000:])
        try:
            running = requests.get('http://localhost:6333', timeout=2).json()
            break
        except requests.RequestException:
            time.sleep(1)
assert running and running['version'] == '1.19.0', 'Qdrant server did not start with the expected version.'
os.environ['QDRANT_PATH'] = ''
os.environ['QDRANT_URL'] = 'http://localhost:6333'
os.environ['QDRANT_API_KEY'] = ''
subprocess.run([sys.executable, '-u', 'upload_vector_index.py'], check=True)
print('Qdrant server ready with the existing 61,068 article vectors.')

## Start one Ollama server per GPU
On two T4 GPUs, each GPU runs its own Qwen, embeddings and BGE worker. Model weights are downloaded once and shared on disk. A single-GPU runtime runs one worker. The startup check rejects substantial CPU offloading or insufficient memory.

In [ ]:
"""Notebook cell: one owned Ollama server per GPU, shared on-disk weights."""
import time
import requests

if not shutil.which('ollama'):
    if not shutil.which('zstd'):
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], check=True)
    subprocess.run(['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'], check=True)

WORKERS = min(2, len(GPUS))
OLLAMA_SERVERS = globals().get('OLLAMA_SERVERS', {})
for index in range(WORKERS):
    host = f'127.0.0.1:{11435 + index}'
    url = f'http://{host}'
    process = OLLAMA_SERVERS.get(index)
    if process is None or process.poll() is not None:
        try:
            requests.get(url + '/api/version', timeout=2).raise_for_status()
        except requests.RequestException:
            pass
        else:
            raise RuntimeError(f'{host} is occupied by an unowned server; restart the runtime.')
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(index), 'OLLAMA_HOST': host,
               'OLLAMA_NUM_PARALLEL': '1', 'OLLAMA_MAX_LOADED_MODELS': '1',
               'OLLAMA_KEEP_ALIVE': '-1'}
        with (REPO / f'ollama-gpu-{index}.log').open('ab') as handle:
            process = subprocess.Popen(['ollama', 'serve'], env=env,
                                       stdout=handle, stderr=subprocess.STDOUT)
        OLLAMA_SERVERS[index] = process
    ready = False
    for _ in range(60):
        if process.poll() is not None:
            raise RuntimeError((REPO / f'ollama-gpu-{index}.log').read_text()[-2000:])
        try:
            requests.get(url + '/api/version', timeout=2).raise_for_status()
            ready = True
            break
        except requests.RequestException:
            time.sleep(1)
    assert ready, f'Ollama GPU {index} did not start; inspect its log.'
    print(f'GPU {index}: dedicated Ollama server {url}')

# Both servers read the same model store; download weights only once.
subprocess.run(['ollama', 'pull', 'qwen3.5:9b'], check=True,
               env={**os.environ, 'OLLAMA_HOST': '127.0.0.1:11435'})
for index in range(WORKERS):
    url = f'http://127.0.0.1:{11435 + index}'
    response = requests.get(url + '/api/tags', timeout=15)
    response.raise_for_status()
    assert any(m['name'] == 'qwen3.5:9b' for m in response.json()['models']), 'Shared Qwen weights unavailable'
print(f'{WORKERS} independent GPU worker(s) ready; combined memory checks run before question processing.')

## Optionally restore Qwen-only checkpoints
If resuming, attach a previously exported `qwen_gpu_workers_top5_results.zip` through Add Input. Only this worker-protocol ZIP is restored; sequential and earlier refinement trials are kept separate. Worker count and all model/runtime settings must match to resume. Matching source, versions, model digest, and GPU are required. Checkpoints under `/kaggle/working` must be preserved through saved outputs or downloading; they do not automatically survive a discarded runtime.

In [ ]:
OUTPUT = REPO / 'results/qwen-gpu-workers-top5'
checkpoint_zip = next(Path('/kaggle/input').rglob('qwen_gpu_workers_top5_results.zip'), None)
if checkpoint_zip and not OUTPUT.exists():
    with zipfile.ZipFile(checkpoint_zip) as archive:
        for name in archive.namelist():
            path = Path(name)
            assert name.startswith('qwen-gpu-workers-top5/') and not path.is_absolute() and '..' not in path.parts and '\\' not in name
        archive.extractall(REPO / 'results')
    print('Trial checkpoint restored.')
print('Trial output:', OUTPUT)

## Run all questions with Qwen only
Qwen makes one local answer call per question; no hosted provider or API key is used. Each worker runs BGE on the original question, returns five documents, then generates an answer. Workers process disjoint questions concurrently and keep isolated checkpoints. Qwen thinking is disabled, context is 8192, and the output cap is 2048. The runner stops on inference or retrieval failure instead of scoring failed calls. Rerun to resume cached model calls and completed questions.

In [ ]:
subprocess.run([sys.executable, '-u', 'benchmark_multi_gpu.py', '--workers', str(WORKERS), '--limit', '0', '--seed', '42',
                '--output', 'results/qwen-gpu-workers-top5'], cwd=REPO, check=True)
from IPython.display import Markdown, display
display(Markdown((OUTPUT / 'comparison.md').read_text()))
print('Review answer_review.csv for citation fidelity and completeness.')

## Export complete or partial results
Run this after completion, or after interrupting to preserve partial work. The exporter merges completed worker records before packaging both merged reports and per-worker checkpoints. Download the ZIP from the Kaggle Files panel and save a notebook version with outputs. Import that ZIP as a Dataset to resume later.

In [ ]:
subprocess.run([sys.executable, 'benchmark_multi_gpu.py', '--workers', str(WORKERS), '--merge-only',
                '--output', 'results/qwen-gpu-workers-top5'], cwd=REPO, check=True)
EXPORT = Path('/kaggle/working/qwen_gpu_workers_top5_results.zip')
with zipfile.ZipFile(EXPORT, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT.rglob('*'):
        if path.is_file() and path.suffix != '.tmp':
            archive.write(path, 'qwen-gpu-workers-top5/' + path.relative_to(OUTPUT).as_posix())
print('Download from Kaggle Files:', EXPORT)